### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

/home/sachin/Desktop/rag-py/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import chromadb

def get_existing_sources_from_disk(persist_directory, collection_name="pdf_documents"):
    """Check ChromaDB directly on disk for already-embedded source filenames"""
    if not os.path.exists(persist_directory):
        return set()
    client = chromadb.PersistentClient(path=persist_directory)
    try:
        collection = client.get_collection(collection_name)
    except Exception:
        return set()
    if collection.count() == 0:
        return set()
    all_data = collection.get(include=["metadatas"])
    sources = set()
    for meta in all_data["metadatas"]:
        src = meta.get("source_file")
        if src:
            sources.add(src)
    return sources

In [3]:
def process_all_pdfs(pdf_directory, existing_sources: set = None):
    """Process all PDF files in a directory, skipping already-embedded ones"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    existing_sources = existing_sources or set()
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files total")
    
    for pdf_file in pdf_files:
        if pdf_file.name in existing_sources:
            print(f"  ⏭ Skipping (already embedded): {pdf_file.name}")
            continue
        
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal NEW documents loaded: {len(all_documents)}")
    return all_documents

VECTOR_STORE_DIR = "../data/vector_store"

existing_sources = get_existing_sources_from_disk(str(VECTOR_STORE_DIR))
print(f"Already embedded PDFs: {existing_sources}")
all_pdf_documents = process_all_pdfs("../data", existing_sources=existing_sources)

Already embedded PDFs: {'agentic-rag.pdf', 'ml_rag_sample.pdf'}
Found 3 PDF files total
  ⏭ Skipping (already embedded): ml_rag_sample.pdf

Processing: predicting-human-behaviour-with-LTSM.pdf
  ✓ Loaded 18 pages
  ⏭ Skipping (already embedded): agentic-rag.pdf

Total NEW documents loaded: 18


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'PDFlib+PDI 9.3.1p2 (C++/Win64)', 'creator': 'PTC Arbortext Layout Developer 12.1.6180/W-x64', 'creationdate': '2022-05-24T10:11:47+05:30', 'source': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'file_path': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'total_pages': 18, 'format': 'PDF 1.6', 'title': 'Predicting human decision making in psychological tasks with recurrent neural networks', 'author': 'Baihan Lin, Djallel Bouneffouf, Guillermo Cecchi', 'subject': '', 'keywords': '', 'moddate': '2022-05-24T10:11:47+05:30', 'trapped': '', 'modDate': "D:20220524101147+05'30'", 'creationDate': "D:20220524101147+05'30'", 'page': 0, 'source_file': 'predicting-human-behaviour-with-LTSM.pdf', 'file_type': 'pdf'}, page_content='RESEARCH ARTICLE\nPredicting human decision making in\npsychological tasks with recurrent neural\nnetworks\nBaihan LinID1,2,3*, Djallel Bouneffouf4, Guillermo Cecchi5\n1 Department of Systems Biology, Columbia University

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[1].page_content[:200]}...")
        print(f"Metadata: {split_docs[1].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 18 documents into 63 chunks

Example chunk:
Content: Abstract
Unlike traditional time series, the action sequences of human decision making usually
involve many cognitive processes such as beliefs, desires, intentions, and theory of mind,
i.e., what oth...
Metadata: {'producer': 'PDFlib+PDI 9.3.1p2 (C++/Win64)', 'creator': 'PTC Arbortext Layout Developer 12.1.6180/W-x64', 'creationdate': '2022-05-24T10:11:47+05:30', 'source': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'file_path': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'total_pages': 18, 'format': 'PDF 1.6', 'title': 'Predicting human decision making in psychological tasks with recurrent neural networks', 'author': 'Baihan Lin, Djallel Bouneffouf, Guillermo Cecchi', 'subject': '', 'keywords': '', 'moddate': '2022-05-24T10:11:47+05:30', 'trapped': '', 'modDate': "D:20220524101147+05'30'", 'creationDate': "D:20220524101147+05'30'", 'page': 0, 'source_file': 'predicting-human-behaviour-with-LTSM.pdf', '

[Document(metadata={'producer': 'PDFlib+PDI 9.3.1p2 (C++/Win64)', 'creator': 'PTC Arbortext Layout Developer 12.1.6180/W-x64', 'creationdate': '2022-05-24T10:11:47+05:30', 'source': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'file_path': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'total_pages': 18, 'format': 'PDF 1.6', 'title': 'Predicting human decision making in psychological tasks with recurrent neural networks', 'author': 'Baihan Lin, Djallel Bouneffouf, Guillermo Cecchi', 'subject': '', 'keywords': '', 'moddate': '2022-05-24T10:11:47+05:30', 'trapped': '', 'modDate': "D:20220524101147+05'30'", 'creationDate': "D:20220524101147+05'30'", 'page': 0, 'source_file': 'predicting-human-behaviour-with-LTSM.pdf', 'file_type': 'pdf'}, page_content='RESEARCH ARTICLE\nPredicting human decision making in\npsychological tasks with recurrent neural\nnetworks\nBaihan LinID1,2,3*, Djallel Bouneffouf4, Guillermo Cecchi5\n1 Department of Systems Biology, Columbia University

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 712.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = None):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory or str("../data/vector_store")
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    
    def get_existing_sources(self) -> set:
        """Return set of source filenames already in the vector store"""
        if self.collection.count() == 0:
            return set()
        all_data = self.collection.get(include=["metadatas"])
        sources = set()
        for meta in all_data["metadatas"]:
            src = meta.get("source_file")
            if src:
                sources.add(src)
        return sources
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 65


In [10]:
chunks

[Document(metadata={'producer': 'PDFlib+PDI 9.3.1p2 (C++/Win64)', 'creator': 'PTC Arbortext Layout Developer 12.1.6180/W-x64', 'creationdate': '2022-05-24T10:11:47+05:30', 'source': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'file_path': '../data/pdf/predicting-human-behaviour-with-LTSM.pdf', 'total_pages': 18, 'format': 'PDF 1.6', 'title': 'Predicting human decision making in psychological tasks with recurrent neural networks', 'author': 'Baihan Lin, Djallel Bouneffouf, Guillermo Cecchi', 'subject': '', 'keywords': '', 'moddate': '2022-05-24T10:11:47+05:30', 'trapped': '', 'modDate': "D:20220524101147+05'30'", 'creationDate': "D:20220524101147+05'30'", 'page': 0, 'source_file': 'predicting-human-behaviour-with-LTSM.pdf', 'file_type': 'pdf'}, page_content='RESEARCH ARTICLE\nPredicting human decision making in\npsychological tasks with recurrent neural\nnetworks\nBaihan LinID1,2,3*, Djallel Bouneffouf4, Guillermo Cecchi5\n1 Department of Systems Biology, Columbia University

In [11]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

if not texts:
    print("No new chunks to embed — skipping embedding & vectorstore update.")
else:
    ## Generate the Embeddings
    embeddings = embedding_manager.generate_embeddings(texts)
    
    ## store into the vector database
    vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 63 texts...


Batches: 100%|██████████| 2/2 [01:25<00:00, 42.98s/it]


Generated embeddings with shape: (63, 384)
Adding 63 documents to vector store...
Successfully added 63 documents to vector store
Total documents in collection: 128


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def get_existing_sources(self) -> set:
        """Return set of source filenames already in the vector store"""
        if self.collection.count() == 0:
            return set()
        all_data = self.collection.get(include=["metadatas"])
        sources = set()
        for meta in all_data["metadatas"]:
            src = meta.get("source") or meta.get("source_file")
            if src:
                sources.add(src)
        return sources

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [13]:
rag_retriever.retrieve("types of machine learning in points")

Retrieving documents for query: 'types of machine learning in points'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_5121c94a_0',
  'content': 'Introduction to Machine Learning\nWhat is Machine Learning?\nMachine learning (ML) is a branch of artificial intelligence that enables systems to learn and improve from\nexperience without being explicitly programmed. Instead of writing rules manually, ML systems identify\npatterns from data and make decisions with minimal human intervention. The term was coined by Arthur\nSamuel in 1959 during his work on a checkers-playing program at IBM.\nTypes of Machine Learning\nThere are three main types of machine learning. Supervised learning uses labeled training data to learn a\nmapping from inputs to outputs. Common examples include spam detection and image classification.\nUnsupervised learning finds hidden patterns in data without labels - clustering and dimensionality reduction\nfall in this category. Reinforcement learning trains an agent to make decisions by rewarding desired\nbehaviors and penalizing undesired ones, commonly used in game-playing

### RAG Pipeline- VectorDB To LLM Output Generation

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [17]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (openai/gpt-oss-120b, llama-3.2-90b-vision-preview, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [18]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [19]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb Context pipeline With LLM output

In [20]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [21]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


### Enhanced RAG Pipeline Features

In [22]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [23]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

Final Answer: No relevant context found.
Summary: The reply indicates that there is no pertinent information available. In other words, the necessary context is missing.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The reply indicates that there is no pertinent information available. In other words, the necessary context is missing.'}


In [24]:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("whos the author of study on agentic rag systems for imporving RAg systems ", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'whos the author of study on agentic rag systems for imporving RAg systems '
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
satisfaction.
2.6 Research Gap
Despite the advancements in both RAG and agentic systems, several gaps remain in the
literature. First, there is a lack of empirical studies that specifically investigate the integration of
agentic capabilities within RA

G frameworks. Most existing research tends to treat these
components in isolation, missing the potential synergies that could arise from their combination.
Additionally, the evaluation methodologies employed in current studies often fail to account for
the dynamic nature of user interactions, leading to a limited understanding of how RAG systems
perform in real-world scenarios.

Strengths:
"What did you like most about the Agentic RAG system?"
reas for Improvement:
"What aspects of the system do you think could be improved?"
Additional Comments:
"Do you have any other comments or suggestions regarding your experience with the system?"

Question: whos the author of study on agentic rag systems for imporving RAg systems 

Answer:

Final Answer: The provided excerpt does not include the name of the author, so the author of the study cannot be identified from the given information.

Citations:
[1] agentic-rag.pdf (page 3)
[2] agentic-rag.pdf (page 17)
Summary: The excerpt contains no autho

In [25]:
import langchain
print("Langchain version:", langchain.__version__)

Langchain version: 1.2.14
